In [2]:
import pandas as pd

# Load the dataset (local CSV previously saved)
df = pd.read_csv('adult_income.csv')

# Clean up whitespace in string columns
for col in df.select_dtypes(include='object'):
    df[col] = df[col].str.strip()

df.head()


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [3]:
from sklearn.model_selection import train_test_split

# Define target and feature set
target = 'income'
train_data, test_data = train_test_split(df, test_size=0.2, random_state=42)

print(f"Train shape: {train_data.shape}, Test shape: {test_data.shape}")


Train shape: (26048, 15), Test shape: (6513, 15)


In [4]:
from autogluon.tabular import TabularPredictor

# Create and train the AutoML predictor
predictor = TabularPredictor(label=target, eval_metric='accuracy').fit(
    train_data,
    time_limit=120  # seconds = 2 minutes
)


/Users/sandilranasinghe/Work/workshops/datastorm/code/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
No path specified. Models will be saved in: "AutogluonModels/ag-20250430_110005"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.2
Python Version:     3.12.8
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 24.1.0: Thu Oct 10 21:06:23 PDT 2024; root:xnu-11215.41.3~3/RELEASE_ARM64_T8132
CPU Count:          10
Memory Avail:       9.47 GB / 24.00 GB (39.4%)
Disk Space Avail:   337.23 GB / 460.43 GB (73.2%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://

In [5]:
# Predict on test set
y_pred = predictor.predict(test_data)
score = predictor.evaluate(test_data)

# View leaderboard
predictor.leaderboard(test_data, silent=True)


/Users/sandilranasinghe/Work/workshops/datastorm/code/venv/lib/python3.12/site-packages/fastai/learner.py:455: UserWarning: load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.
If you only need to load model weights and optimizer state, use the safe `Learner.load` instead.
  warn("load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.\nIf you only need to load model weights and optimizer state, use the safe `Learner.load` instead.")


,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,CatBoost,0.879625,0.8752,accuracy,0.010897,0.003048,10.405194,0.010897,0.003048,10.405194,1,True,7
1,WeightedEnsemble_L2,0.879625,0.8752,accuracy,0.011505,0.003688,10.467925,0.000608,0.000640,0.062731,2,True,14
2,XGBoost,0.877936,0.8724,accuracy,0.014935,0.009977,0.924178,0.014935,0.009977,0.924178,1,True,11
3,LightGBM,0.876094,0.8688,accuracy,0.006936,0.004952,0.601483,0.006936,0.004952,0.601483,1,True,4
4,LightGBMXT,0.875019,0.8684,accuracy,0.035566,0.026885,2.162210,0.035566,0.026885,2.162210,1,True,3
5,LightGBMLarge,0.874559,0.8700,accuracy,0.009067,0.004269,2.120936,0.009067,0.004269,2.120936,1,True,13
6,RandomForestEntr,0.865193,0.8524,accuracy,0.114678,0.038105,0.786419,0.114678,0.038105,0.786419,1,True,6
7,RandomForestGini,0.861508,0.8528,accuracy,0.108201,0.038048,3.529132,0.108201,0.038048,3.529132,1,True,5
8,NeuralNetFastAI,0.860740,0.8576,accuracy,0.049400,0.014022,8.518632,0.049400,0.014022,8.518632,1,True,10
9,NeuralNetTorch,0.858591,0.8520,accuracy,0.018903,0.008239,9.672961,0.018903,0.008239,9.672961,1,True,12


In [8]:
# Show feature importance
importance_df = predictor.feature_importance(test_data)

importance_df.head(n=20)

Computing feature importance via permutation shuffling for 14 features using 5000 rows with 5 shuffle sets...
	1.27s	= Expected runtime (0.25s per shuffle set)
	0.42s	= Actual runtime (Completed 5 of 5 shuffle sets)


,importance,stddev,p_value,n,p99_high,p99_low
capital_gain,0.05076,0.003201,0.000002,5,0.057351,0.044169
relationship,0.02236,0.002651,0.000023,5,0.027819,0.016901
occupation,0.02140,0.002627,0.000027,5,0.026809,0.015991
age,0.02028,0.003354,0.000087,5,0.027187,0.013373
capital_loss,0.01448,0.000934,0.000002,5,0.016403,0.012557
marital_status,0.01228,0.002194,0.000117,5,0.016797,0.007763
education_num,0.00900,0.001606,0.000117,5,0.012307,0.005693
hours_per_week,0.00484,0.001431,0.000819,5,0.007787,0.001893
workclass,0.00472,0.002361,0.005532,5,0.009580,-0.000140
education,0.00364,0.002076,0.008613,5,0.007914,-0.000634
